**[🏠 Course Home](../README.md)** | **📖 [Conceptual Track](../conceptual/16_bayesian_decision_theory_and_predictive_checks.ipynb)** | **🤖 [AI Track](16_bayesian_decision_theory_and_predictive_checks.ipynb)** | **🐍 [Python Track](../python/16_bayesian_decision_theory_and_predictive_checks.ipynb)** | **🔵 [R Track](../r/16_bayesian_decision_theory_and_predictive_checks.ipynb)** | **⬅️ [Prev: Ch 15: Bayesian Decay & Memory](15_calibrating_bayesian_decay_and_memory.ipynb)** | **➡️ [Next: Ch 17: Tolerance Intervals](17_tolerance_intervals_and_decision_sizing.ipynb)**

*🧭 Part IV: Dynamic Systems & Decision Theory — Chapter 16 of 22*

---

# Chapter 16: Decision Theory & Predictive Checks
### *Asymmetric Loss Matrices, Universal Thresholds, and Sanity Mirrors*

---

> **🧭 Course Context & Why This Module Matters in the AI Era:**
>
> A business cannot deploy a probability density function. An automated
> infrastructure robot cannot page an on-call engineer with a continuous
> posterior curve. At 3:00 AM, an autonomous system must execute a discrete,
> irrevocable action:
> * *Ship the release to production, or abort and trigger a rollback?*
> * *Quarantine a suspicious test, or allow it to block developer PRs?*
> * *Sound the fire alarm and evacuate the building, or suppress the signal?*
>
> When engineers prompt Large Language Models (LLMs) to automate operational
> decisions, AI assistants default to naive, symmetric decision heuristics:
> 1. **The 50% Coin-Flip Rule**: Recommending action only if the posterior
>    probability exceeds $0.50$.
> 2. **Arbitrary Significance Thresholds**: Suggesting inaction because $p >
>    0.05$ or confidence intervals span zero.
>
> In real engineering systems, consequences are radically asymmetric: a False
> Negative (shipping software that causes a major outage) costs orders of
> magnitude more than a False Positive (briefly quarantining a healthy test).
>
> This module teaches you how to prompt AI agents to bind posterior beliefs to
> explicit economic loss matrices and validate generative model sanity using
> Posterior Predictive Checks (PPC).


## Part 1: The Naive Prompt Trap & Forensic Post-Mortem

### The Prompt That Fools the AI

A continuous integration (CI) team manages an automated pipeline. A critical
test, `test_payment_gateway`, has failed $k = 2$ times in $n = 27$ runs
($7.4\%$ raw failure rate). The team must decide whether to automatically
quarantine the test (allowing PR merges to proceed while devs investigate) or
ship the release immediately.

They prompt an AI assistant:

```text
Our payment gateway integration test failed 2 times out of 27 runs (7.4%
failure rate). We need an automated Python rule: should we quarantine this test
or ship the release? Calculate the posterior failure probability and recommend
the optimal action.
```

---

### The Plausible AI Hallucination

The AI sets up a conjugate Beta-Binomial model with prior $\text{Beta}(1, 1)$:
$$\alpha_{\text{post}} = 1 + 2 = 3, \qquad \beta_{\text{post}} = 1 + 25 = 26$$
The posterior mean failure rate is:
$$\mathbb{E}[\theta \mid \mathcal{D}] = \frac{3}{29} \approx 10.3\%$$

The AI provides this recommendation:

```text
Decision Analysis:
1. Posterior Mean Failure Rate: 10.3%
2. Probability that Failure Rate exceeds 20%: 8.2%
3. Classical p-value against null hypothesis (p <= 0.05): p = 0.23 (not significant)

RECOMMENDATION: SHIP THE RELEASE. DO NOT QUARANTINE.
Because the estimated failure probability (10.3%) is well below the standard 50%
decision threshold, and the 95% credible interval [2.2%, 24.1%] overlaps with
acceptable noise, quarantining would unnecessarily disrupt engineering velocity.
```

---

### Forensic Post-Mortem: Why the System Crashes

```
           THE ASYMMETRIC LOSS DISASTER: THE SMOKE ALARM
 Expected
   Loss
  $2,500 |               * [Action: Ship]
         |              /  Expected Loss = $2,069!
  $2,000 |             /   (Exposing company to $20k outage risk)
         |            /
  $1,500 |           /
         |          /
  $1,000 |         /     ● [Action: Quarantine]
         |        /        Expected Loss = $896 (OPTIMAL ACTION!)
    $500 |       /
         +------+------------------------------------------
             p = 0.0476 (Universal Threshold c*)    p = 0.103 (Posterior Mean)
```

#### 1. The Catastrophic Asymmetry of Real Decisions
The AI's recommendation caused the company to ship broken software. The payment
service crashed in production, resulting in $\$20,000$ in lost transactions and
emergency engineering triage:
$$\text{Cost of False Negative } (L_{\text{FN}}) = \$20,000$$
Had the system quarantined the test, an engineer would have diagnosed a mock
race condition in 30 minutes:
$$\text{Cost of False Positive } (L_{\text{FP}}) = \$1,000$$

#### 2. The Universal Decision Threshold ($c^*$)
Under Bayesian Decision Theory, the optimal action minimizes **Expected Loss**:
$$\mathbb{E}[L(\text{Ship})] = P(\text{Broken} \mid \mathcal{D}) \times L_{\text{FN}} = 0.1034 \times \$20,000 = \mathbf{\$2,068.96}$$
$$\mathbb{E}[L(\text{Quarantine})] = P(\text{Healthy} \mid \mathcal{D}) \times L_{\text{FP}} = (1 - 0.1034) \times \$1,000 = \mathbf{\$896.55}$$

The threshold where Quarantining becomes economically dominant is:
$$c^* = \frac{L_{\text{FP}}}{L_{\text{FP}} + L_{\text{FN}}} = \frac{1000}{1000 + 20000} = \frac{1}{21} \approx \mathbf{4.76\%}$$
Because the posterior probability of failure ($10.3\%$) exceeds $4.76\%$,
**Quarantining is mathematically mandatory**! The AI's symmetric $50\%$ advice
cost the business over $\$1,170$ in expected risk per decision.

#### 3. The Skipped Sanity Mirror: Posterior Predictive Checks (PPC)
The AI failed to check whether the underlying Beta-Binomial model was even
valid. In the raw test logs, the 2 failures occurred in **immediate succession**
(Runs 14 and 15), representing a burst streak. An i.i.d. Binomial model cannot
generate burst dynamics ($p_{\text{ppp}} = 0.008$), proving that the generative
model was misspecified.


## Part 2: Prompting as Probabilistic Programming — Declarative Specifications

To eliminate AI statistical hallucinations, we must specify the generative
physics, prior distributions, and decision loss rather than asking vague
questions. Depending on your workflow, you can specify models using three
progressively simpler patterns:

---

### Level 1: The Everyday 3-Sentence Micro-Spec *(Fastest for Chat)*
Copy and paste these 3 plain sentences into any LLM:

```markdown
1. State: Latency failure probability theta in [0, 1]; observed k = 2 failures
   in n = 27 trials.
2. Loss Matrix: False Negative (shipping broken code) costs L_FN = $20,000;
   False Positive (unnecessary quarantine) costs L_FP = $1,000.
3. Action: Compute universal decision threshold c* = L_FP / (L_FP + L_FN) = 4.76%
   and quarantine if P(theta > 0.05 | D) > c*.
```

---

### Level 2: The Model Formula Shorthand *(Concise Probabilistic DSL)*
For quick prompt engineering with statistical models:

```text
Loss:      L(a_ship, theta) = L_FN * theta,  L(a_quar, theta) = L_FP * (1 - theta)
Threshold: c* = L_FP / (L_FP + L_FN) = 1000 / 21000 = 4.76%
Optimal:   Quarantine iff P(theta > 0.05 | D) > 0.0476
```

---

### Level 3: The Socratic Interviewer Meta-Prompt *(Zero Effort)*
If you do not want to write the spec yourself, force the AI to interview you:

> *"Act as a Decision Theorist. We need to decide whether to ship a build with 2
failures in 27 runs. Ask me 3 questions about business outage costs, quarantine
delay costs, and risk tolerance before making a recommendation."*

---

<details>
<summary><b>📋 View Full Production YAML Contract (For Automated CI/CD Pipelines & Autonomous Agents)</b></summary>

```yaml
declarative_probabilistic_specification:
  module: "bayesian_decision_theory_and_predictive_checks"
  domain: "continuous_integration_release_governance"
  
  decision_space:
    actions:
      - name: "Action_Ship"
        description: "Promote build to production deployment"
      - name: "Action_Quarantine"
        description: "Isolate test suite, pause release, alert test owner"

  state_space:
    states:
      - name: "State_Healthy"
        condition: "True latent failure rate theta <= 0.02"
      - name: "State_Broken"
        condition: "True latent failure rate theta > 0.02"

  asymmetric_loss_matrix_usd:
    Action_Ship:
      State_Healthy: 0.0
      State_Broken: 20000.0       # False Negative: Catastrophic production outage
    Action_Quarantine:
      State_Healthy: 1000.0       # False Positive: Developer inspection time
      State_Broken: 0.0           # True Positive: Outage averted

  computational_invariants:
    universal_threshold: "c_star = L_FP / (L_FP + L_FN) = 1000 / 21000 = 0.0476"
    optimal_action_rule: "Choose Action_Quarantine if P(theta > 0.02 | Data) >= c_star"

  posterior_predictive_checks:
    replications: 2000
    test_statistics:
      - name: "max_consecutive_failures"
        formula: "max_streak(y_sim)"
      - name: "dispersion_index"
        formula: "var(y_sim) / mean(y_sim)"
    validity_bounds:
      bayesian_p_value_range: [0.05, 0.95]
      on_violation: "Reject model; signal burst/flaky temporal dependence"
```

</details>


## Part 3: The 4 Mandatory Agentic Verification Gates

Before trusting any autonomous decision agent, verify these four gates:

```
        THE 4 MANDATORY AGENTIC VERIFICATION GATES (CH 16)
 ┌───────────────────────┐      ┌─────────────────────────┐
 │ Gate 1: Loss Matrix   │ ───► │ Gate 2: Expected Loss   │
 │ L_FP > 0, L_FN > 0    │      │ Action minimizes        │
 │ c* in (0, 1) defined  │      │ ∫ L(a, θ) p(θ|D) dθ     │
 └───────────────────────┘      └─────────────────────────┘
             │                               │
             ▼                               ▼
 ┌───────────────────────┐      ┌─────────────────────────┐
 │ Gate 3: PPC Mirror    │ ───► │ Gate 4: Cost Perturb    │
 │ 0.05 <= p_ppp <= 0.95 │      │ Optimal action stable   │
 │ Streak stat checked   │      │ under +/- 30% cost noise│
 └───────────────────────┘      └─────────────────────────┘
```

### Gate 1: Loss Matrix Completeness & Invariant Gate
* **Objective**: Confirm that decision loss is explicitly parameterized.
* **Criterion**: Verify that the loss matrix defines non-zero costs for all
  incorrect action-state pairs ($L_{\text{FP}} > 0, L_{\text{FN}} > 0$).
  Compute the universal decision threshold:
  $$c^* = \frac{L_{\text{FP}}}{L_{\text{FP}} + L_{\text{FN}}}$$
  Assert that $0.001 \le c^* \le 0.50$. Reject any prompt or code defaulting
  to symmetric $50\%$ thresholds.

### Gate 2: Expected Loss Minimization Gate
* **Objective**: Guarantee that the selected action minimizes risk.
* **Criterion**: Compute expected loss across all actions $a \in \mathcal{A}$:
  $$\mathbb{E}[L(a)] = \int L(a, \theta) p(\theta \mid \mathcal{D}) \, d\theta$$
  Assert that the recommended action $a^*$ satisfies:
  $$\mathbb{E}[L(a^*)] = \min_{a} \mathbb{E}[L(a)]$$

### Gate 3: Posterior Predictive Retrodiction Gate (The Sanity Mirror)
* **Objective**: Audit whether generative model retrodicts extreme data.
* **Criterion**: Simulate 2,000 synthetic datasets
  $y^{\text{rep}} \sim p(y \mid \mathcal{D})$.
  Calculate test statistics (e.g., maximum run streak of failures $T(y)$).
  Compute the Posterior Predictive p-Value:
  $$p_{\text{ppp}} = P\left(T(y^{\text{rep}}) \ge T(y^{\text{obs}}) \mid \mathcal{D}\right)$$
  Assert that $0.05 \le p_{\text{ppp}} \le 0.95$. If $p_{\text{ppp}} < 0.05$,
  flag the model as misspecified (e.g., burst failure clustering).

### Gate 4: Cost Sensitivity Robustness Gate
* **Objective**: Ensure decisions stay stable under cost estimation noise.
* **Criterion**: Perturb $L_{\text{FN}}$ and $L_{\text{FP}}$ by $\pm 30\%$.
  Verify that the recommended action $a^*$ remains invariant across the cost
  envelope.


## Part 4: Worked Agent Execution Transcript & Telemetry

### Problem Context: `test_payment_gateway` Quarantine Decision
* Dataset: $n = 27$ runs, $k = 2$ failures.
* Prior: $\text{Beta}(\alpha_0 = 1.0, \beta_0 = 1.0)$.
* Posterior: $\text{Beta}(\alpha_n = 3.0, \beta_n = 26.0)$.
* Posterior probability of regression:
  $P(\theta > 0.02 \mid \mathcal{D}) = 0.932$ ($93.2\%$).

```text
================================================================================
              BAYESIAN DECISION & PPC RETRODICTION TELEMETRY LOG
================================================================================
Target Entity: test_payment_gateway | Observed Runs: 27 | Observed Failures: 2
Posterior: Beta(3.0, 26.0) | Mean Latent Failure Rate: 10.34% +/- 5.51%
--------------------------------------------------------------------------------
ECONOMIC RISK ANALYSIS:
  False Negative Cost (Ship Broken):        $20,000.00
  False Positive Cost (Quarantine Healthy):  $1,000.00
  Universal Decision Threshold c*:               0.0476 (4.76% probability)

EXPECTED LOSS CALCULATIONS:
  Action [Ship Build]:        E[Loss] = 0.1034 * $20,000 = $2,068.96
  Action [Quarantine Test]:   E[Loss] = 0.8966 *  $1,000 =   $896.55  <-- OPTIMAL!

FINANCIAL ARBITRAGE:
  Quarantining saves $1,172.41 in expected economic risk per execution!
  Recommendation: QUARANTINE TEST AND BLOCK RELEASE.
--------------------------------------------------------------------------------
POSTERIOR PREDICTIVE CHECK (PPC) SANITY AUDIT:
  Observed Max Consecutive Failure Streak:  2 consecutive failures (Runs 14-15)
  Simulated Mean Max Streak across 2000 reps: 1.04 streaks
  Simulated Max Streak Distribution:
    P(Max Streak = 1) = 96.2%
    P(Max Streak >= 2) =  3.8%
  Posterior Predictive p-value (p_ppp): 0.038  <-- CRITICAL WARNING!
  
DIAGNOSTIC NOTICE:
  p_ppp = 0.038 < 0.05 indicates model failure! Observed failures occur in
  correlated temporal bursts rather than independent Bernoulli trials.
  Recommend upgrading from i.i.d. Beta-Binomial to Markov-Switching Beta model.
================================================================================
```

---

### Visualization: The Asymmetric Decision Boundary

```
          ASYMMETRIC DECISION BOUNDARY & EXPECTED LOSS
 Expected
   Loss
  $2,500 |                              * [Ship Build: $2,069]
         |                             /
  $2,000 |                            /
         |                           /
  $1,500 |                          /
         |                         /
  $1,000 |------------------------/----● [Quarantine Test: $896]
         |                       /
    $500 |                      /
         +---------------------+-----------------------------
         0.00                0.0476                        0.25
                         Threshold c*
         [--- SHIP REGION ---] [---- QUARANTINE REGION -----]
```


## Part 5: Naive AI Prompt vs. Calibrated Epistemic Specification

| Architectural Aspect | Naive AI Assistant Output | Calibrated Probabilistic Specification |
| :--- | :--- | :--- |
| **Decision Heuristic** | Symmetric 50% cutoff or frequentist $p < 0.05$ | Expected Loss Minimization via Loss Matrix |
| **Cost Awareness** | Completely blind to economic consequences | Quantifies $L_{\text{FN}} = \$20\text{k}$ vs $L_{\text{FP}} = \$1\text{k}$ explicitly |
| **Universal Threshold**| $c^* = 0.50$ (causes $\$2,000$ risk exposure) | $c^* = \frac{L_{\text{FP}}}{L_{\text{FP}}+L_{\text{FN}}} = 4.76\%$ |
| **Decision Recommendation**| SHIP (allows broken release into production) | QUARANTINE (saves $\$1,172$ in expected risk) |
| **Model Verification** | Zero sanity auditing | Posterior Predictive Checks (PPC) with test statistics |
| **Temporal Bursts** | Misses correlated failure clusters | Catches burst dynamics via $p_{\text{ppp}} = 0.038 < 0.05$ |


## Part 6: 🎯 Check Your Understanding — 3-Tier "Audit the AI" Exercises

Test your skills in auditing AI-generated decision pipelines.

---

### 🟢 Tier 1: Spot the AI Hallucination
An AI agent analyzes an automated anomaly detection monitor:
> *"The posterior probability of a database hardware failure is 3.5%.
> Because 3.5% is far below 50%, we recommend suppressing the pager alert to
> prevent engineer alert fatigue."*

Hardware failure causes an unrecoverable data loss event costing $\$100,000$.
Paging the on-call engineer costs $\$150$ in engineer time.

**Task**: Identify the flaw and compute the true optimal decision threshold.

<details>
<summary><b>Click here to view the solution...</b></summary>

**Solution**:
The agent used a symmetric $50\%$ decision cutoff on an ultra-asymmetric
infrastructure event.

The Universal Decision Threshold is:
$$c^* = \frac{L_{\text{FP}}}{L_{\text{FP}} + L_{\text{FN}}} = \frac{150}{150 + 100000} \approx \mathbf{0.15\%}$$
Because the posterior probability of failure ($3.5\%$) is more than **23 times
higher** than $c^* = 0.15\%$, suppressing the alert is catastrophic negligence.
The system must page the engineer immediately!
</details>

---

### 🟡 Tier 2: Refactor the Flawed Specification
An engineer drafted this decision rule in Python:

```python
# Flawed AI snippet
def release_gate(posterior_samples):
    p95 = np.percentile(posterior_samples, 95)
    if p95 > 0.05:
        return "REJECT_RELEASE"
    return "SHIP"
```

**Task**: Rewrite this into a formal loss-minimizing decision function that
takes arbitrary cost parameters $L_{\text{FN}}$ and $L_{\text{FP}}$.

<details>
<summary><b>Click here to view the solution...</b></summary>

**Solution**:

```python
import numpy as np

def calibrated_bayesian_decision(posterior_samples, cost_fn: float, cost_fp: float, threshold_theta: float = 0.02):
    # Expected loss minimization decision function
    # cost_fn: False Negative cost (shipping broken software)
    # cost_fp: False Positive cost (unnecessary quarantine)
    # 1. Compute posterior probability of broken state
    prob_broken = np.mean(posterior_samples > threshold_theta)
    
    # 2. Compute Universal Decision Threshold
    c_star = cost_fp / (cost_fp + cost_fn)
    
    # 3. Compute Expected Losses explicitly
    expected_loss_ship = prob_broken * cost_fn
    expected_loss_quarantine = (1.0 - prob_broken) * cost_fp
    
    # 4. Optimal Decision Rule
    action = "QUARANTINE" if prob_broken >= c_star else "SHIP"
    
    return {
        "action": action,
        "prob_broken": float(prob_broken),
        "threshold_c_star": float(c_star),
        "expected_loss_ship": float(expected_loss_ship),
        "expected_loss_quarantine": float(expected_loss_quarantine),
        "arbitrage_savings": float(abs(expected_loss_ship - expected_loss_quarantine))
    }
```
</details>

---

### 🔴 Tier 3: Architect an Autonomous Agent Contract
You are designing an autonomous smoke-alarm release agent for a medical device
gateway. If a patient telemetry monitor drops packets, the event is fatal
($L_{\text{FN}} = \$1,000,000$). A false alarm triggers an ambulance dispatch
($L_{\text{FP}} = \$2,000$). The agent must compute posterior risk and run a
posterior predictive check on packet inter-arrival times.

**Task**: Write the complete declarative specification and audit contract.

<details>
<summary><b>Click here to view the solution...</b></summary>

**Solution**:

```yaml
autonomous_agent_contract:
  role: "Critical Healthcare Gateway Guardian"
  objective: "Loss-optimal alarm dispatch with generative model verification"
  
  loss_matrix:
    false_negative_cost_usd: 1000000.0  # Fatal outcome
    false_positive_cost_usd: 2000.0     # Dispatch fee
    universal_threshold: "c* = 2000 / (1000000 + 2000) = 0.001996 (~0.20%)"
    
  decision_rule:
    condition: "Trigger emergency alarm if P(packet_drop_rate > 0.001 | Data) >= 0.002"
    
  posterior_predictive_sanity_audit:
    replications: 5000
    test_statistic: "max_inter_arrival_gap_ms"
    validity_rule:
      p_value_bounds: [0.05, 0.95]
      action_on_failure:
        - "Flag telemetry model failure: burst packet loss detected"
        - "Escalate to redundant cellular backup channel"
        - "Log diagnostic telemetry trace"
```
</details>


## Part 7: Key Takeaways & Navigation

### Core Architectural Rules for Bayesian Decision Systems
1. **Never Make Decisions on Probabilities Alone**: A probability is not an
   action. Every production decision requires multiplying posterior beliefs by
   an economic loss matrix.
2. **Master the Universal Decision Threshold ($c^*$)**: Whenever costs are
   asymmetric, the threshold is $c^* = L_{\text{FP}} / (L_{\text{FP}} +
   L_{\text{FN}})$. Never accept arbitrary $50\%$ cutoffs or $p < 0.05$ rules.
3. **Always Run the Sanity Mirror (PPC)**: Even a mathematically optimal
   decision engine fails if the underlying model is misspecified. Always
   verify that the generative model can retrodict observed extreme streaks.
4. **Audit Robustness under Cost Uncertainty**: Verify that the selected action
   remains dominant across $\pm 30\%$ shifts in estimated financial impact.

---

### 🧭 Course Navigation

* 🏠 **[Course Home & Master Syllabus](../README.md)**
* ⬅️ **Previous**: **[Ch 15: Dynamic Memory & Decay (γ)](15_calibrating_bayesian_decay_and_memory.ipynb)** *(Part IV: Dynamic Systems, Decision Theory & Predictive Guarantees)*
* ➡️ **Next**: **[Ch 17: Tolerance Intervals & Sizing](17_tolerance_intervals_and_decision_sizing.ipynb)** *(Part IV: Dynamic Systems, Decision Theory & Predictive Guarantees)*
* 🔄 **Companion Tracks for Chapter 16**: [📖 Conceptual](../conceptual/16_bayesian_decision_theory_and_predictive_checks.ipynb) | [🤖 AI Track](16_bayesian_decision_theory_and_predictive_checks.ipynb) | [🐍 Python](../python/16_bayesian_decision_theory_and_predictive_checks.ipynb) | [🔵 R](../r/16_bayesian_decision_theory_and_predictive_checks.ipynb)
